# RealSaS — Frozen Semantic Evidence / Knight V1

Bu notebook **IRIS geometry'yi fit etmez, değiştirmez veya yeniden çalıştırmaz**.

Amaç:

`8 source view → frozen open-vocabulary semantics → canonical M lift → grip / behavior / source-overlap / posed order-conflict diagnostics`

Canonical geometry ve recorded Knight M poses read-only baseline'dır. External modeller yalnız evidence producer'dır.


In [ ]:
!nvidia-smi || true
!rm -rf /content/RealSaS-OPT
!git clone --depth 1 --branch research/iris-semantic-behavior-v1-20261012 https://github.com/merynz/RealSaS-OPT.git /content/RealSaS-OPT
%cd /content/RealSaS-OPT
!pip -q install -r requirements/semantic-research.txt


In [ ]:
import torch, json, pathlib
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Bu oracle run GPU ile çok daha pratik."


## 1) Frozen semantic proposals

İlk Knight oracle'ında taxonomy'yi özellikle dar tutuyoruz. Başarı görürsek genişletiriz.

Grounding model: `IDEA-Research/grounding-dino-tiny` (pinned revision)  
Mask model: `facebook/sam2.1-hiera-small` (pinned revision)

**Hiçbiri canonical truth değildir.**


In [ ]:
OUT = pathlib.Path("/content/realsas_semantic_v1")
OUT.mkdir(exist_ok=True)
LABELS = "body,head,arm,forearm,hand,cape,cloak,armor,helmet,horn,sword,weapon,shield,prop"

!python tools/research/run_frozen_semantic_evidence_v1.py \
  --source-zip tests/fixtures/single_m_caa_knight_v1/source_rgba.zip \
  --output {OUT}/semantic_masks.npz \
  --labels "{LABELS}" \
  --detection-threshold 0.16 \
  --max-per-label 2 \
  --device cuda


## 2) Aynı evidence'ı canonical M'ye lift et

Bu adım geometry inference yapmaz. Sealed Knight rest M + recorded poses kullanılır.

Çıktılar:
- subject-local semantic owner candidate'ları
- rigid / articulated / secondary-deformable behavior evidence
- hand↔weapon `GRIP_CANDIDATE`
- source-observed `FRONT_OF_SOURCE_EVIDENCE`
- run motion sırasında `POSE_ORDER_CONFLICT`


In [ ]:
!python tools/research/compile_knight_semantic_relations_v1.py \
  --semantic-evidence {OUT}/semantic_masks.npz \
  --output {OUT}/knight_semantic_relations.json \
  --diagnostic-resolution 256


In [ ]:
report = json.loads((OUT/"knight_semantic_relations.json").read_text())
print("M unchanged:", report["rest_geometry_byte_equal_after"])
print("recorded poses unchanged:", report["recorded_pose_byte_equal_after"])
print("qualified semantic faces:", report["qualified_semantic_face_count"], "/", report["face_count"])
print("owners:", len(report["owners"]))
print("grip candidates:", len(report["grip_candidates"]))
print("source overlap relations:", len(report["source_overlap_relations"]))
print("pose order conflicts:", len(report["pose_order_conflicts"]))

print("\nTop owners")
for row in sorted(report["owners"], key=lambda x:-x["face_count"])[:30]:
    print(row)

print("\nTop grip candidates")
for row in sorted(report["grip_candidates"], key=lambda x:-x["confidence"])[:20]:
    print(row)

print("\nTop order conflicts")
for row in sorted(report["pose_order_conflicts"], key=lambda x:-x["confidence"])[:30]:
    print(row)


## 3) Ham semantic mask'leri gör

Bu görsel yalnız perception quality kontrolüdür; Compiler qualification değildir.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, zipfile, io
from PIL import Image
from models.iris.semantic_evidence_v1 import load_mask_evidence_v1

ev = load_mask_evidence_v1(OUT/"semantic_masks.npz")
with zipfile.ZipFile("tests/fixtures/single_m_caa_knight_v1/source_rgba.zip") as z:
    sources = [Image.open(io.BytesIO(z.read(f"observations/V{v}.png"))).convert("RGB") for v in range(8)]

for v in range(8):
    ids = np.where(ev.proposal_view_index == v)[0]
    if len(ids) == 0:
        continue
    fig = plt.figure(figsize=(12, 6))
    plt.imshow(sources[v])
    for pi in ids:
        mask = ev.proposal_masks[pi]
        ys, xs = np.where(mask)
        if len(xs) == 0: continue
        plt.contour(mask.astype(float), levels=[0.5], linewidths=1)
        plt.text(xs.mean(), ys.mean(), f"{ev.label_for(pi)} {ev.proposal_score[pi]:.2f}", fontsize=8)
    plt.title(f"V{v} — external modal semantic evidence")
    plt.axis("off")
    plt.show()


## Karar kapısı

İlk run'da özellikle şunlara bak:

1. `cape/cloak` ve `arm` owner'ları gerçekten doğru M bölgelerine lift olmuş mu?
2. `hand` + `sword/weapon` doğru ayrılmış mı ve grip candidate mantıklı mı?
3. V4/V5 civarında source overlap relation oluşmuş mu?
4. Bilinen elbow/cape run anlarında `POSE_ORDER_CONFLICT` çıkıyor mu?
5. `cape/cloak` behavior evidence `SECONDARY_DEFORMABLE`, sword/shield `RIGID` çıkıyor mu?

Bunlar güçlü ise sonraki PR'da **typed qualification + mevcut single-M runtime consumer** bağlanır. Zayıf ise geometry'yi değiştirmiyoruz; provider/prompt/cross-view association katmanını geliştiriyoruz.


In [ ]:
# Sonuçları tek dosyada indirilebilir paketle.
!cd {OUT} && zip -q semantic_knight_v1_results.zip semantic_masks.npz knight_semantic_relations.json
print(OUT/"semantic_knight_v1_results.zip")
